# 02 — Product-by-week demand panel

**Purpose:** convert cleaned invoice lines into a regular weekly time series for each product.

**Input:** `../data/processed/clean_transactions.csv`  
**Output:** `../data/processed/weekly_panel.csv`

The aggregation creates one row per `StockCode` and Monday-starting week. Units are summed, orders are counted as distinct invoices, and average price is calculated from transaction prices. The first and last partial weeks are excluded.

For weeks in which a product has no transaction, units and order count are filled with zero. Average price is forward-filled from the last known selling price; a missing weekly price does not mean the product was sold at zero price.

In [1]:
import pandas as pd

clean = pd.read_csv("../data/processed/clean_transactions.csv",
                    parse_dates=["InvoiceDate"], dtype={"StockCode": str})
print(clean.shape)

(1036877, 7)


In [2]:
# Assign each invoice date to the Monday starting its week
clean["week"] = clean["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
print(clean["week"].min(), "to", clean["week"].max())

2009-11-30 00:00:00 to 2011-12-05 00:00:00


In [3]:
# Remove the first and last weeks because they are incomplete
first_week = clean["week"].min()
last_week = clean["week"].max()
clean = clean[(clean["week"] > first_week) & (clean["week"] < last_week)]
print(clean["week"].min(), "to", clean["week"].max())

2009-12-07 00:00:00 to 2011-11-28 00:00:00


In [4]:
# Aggregate transaction lines to one product-week row 
weekly = (clean.groupby(["StockCode", "week"])
               .agg(units=("Quantity", "sum"),
                    orders=("Invoice", "nunique"),
                    avg_price=("Price", "mean"))
               .reset_index())
print(weekly.shape)
weekly.head()

(191530, 5)


,StockCode,week,units,orders,avg_price
0,10002,2009-12-07,24,4,1.062500
1,10002,2009-12-14,37,3,1.133333
2,10002,2009-12-21,14,3,1.133333
3,10002,2010-01-04,15,2,0.850000
4,10002,2010-01-11,191,4,0.850000


In [5]:
# Create a complete weekly calendar for each product
end = weekly["week"].max()
first_sale = weekly.groupby("StockCode")["week"].min()

grid = pd.concat([
    pd.DataFrame({"StockCode": code,
                  "week": pd.date_range(start, end, freq="7D")})
    for code, start in first_sale.items()
], ignore_index=True)

print(grid.shape)

(407702, 2)


In [6]:
# Merge observed sales onto the calendar and fill unobserved sales with zero 
panel = grid.merge(weekly, on=["StockCode", "week"], how="left")
panel[["units", "orders"]] = panel[["units", "orders"]].fillna(0)

# price: carry the last observed price forward using past information
panel = panel.sort_values(["StockCode", "week"])
panel["avg_price"] = panel.groupby("StockCode")["avg_price"].ffill()

print(panel.shape)
print(panel.isna().sum())


(407702, 5)
StockCode    0
week         0
units        0
orders       0
avg_price    0
dtype: int64


In [7]:
# Save the completed weekly panel 
panel.to_csv("../data/processed/weekly_panel.csv", index=False)
print(panel["StockCode"].nunique(), "products,", panel["week"].nunique(), "weeks")

4675 products, 104 weeks


## Weekly-panel validation

The checks confirm a unique product-week key, a seven-day interval within each product history, and no missing values in the five panel columns. For the supplied cleaned dataset, the expected panel contains 407,702 rows, 4,675 products and 104 weeks.

In [8]:
assert not panel.duplicated(["StockCode", "week"]).any(), "Duplicate product-week rows found"
weekly_gaps = panel.sort_values(["StockCode", "week"]).groupby("StockCode")["week"].diff().dropna()
assert (weekly_gaps == pd.Timedelta(days=7)).all(), "Irregular weekly interval found"
assert panel[["StockCode", "week", "units", "orders", "avg_price"]].notna().all().all(), "Missing panel values found"

print("Rows:", f"{len(panel):,}")
print("Products:", f"{panel['StockCode'].nunique():,}")
print("Weeks:", f"{panel['week'].nunique():,}")
print("Weekly-panel assertions passed.")

Rows: 407,702
Products: 4,675
Weeks: 104
Weekly-panel assertions passed.
